In [2]:
df_admissions = spark.read.table("Bronze_LH.dbo.bronze_admissions")

display(df_admissions)

silver_patients= spark.read.table("silver_patients")

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 74cc0c61-be57-4144-ab4e-351d0fd66f8f)

### For Incremental data


In [3]:
from pyspark.sql.functions import col, desc, row_number
from pyspark.sql.window import Window

spec = Window.partitionBy("AdmissionID").orderBy(
    col("LastModifiedDate").desc()

)
df_admissions = (
    df_admissions
                .withColumn("rn",row_number().over(spec))
                .filter(col("rn")==1)
                .drop("rn")
)


StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 5, Finished, Available, Finished, False)

In [4]:

df_admissions.printSchema()

print("columns:",len(df_admissions.columns))
print("Rows:",df_admissions.count())

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 6, Finished, Available, Finished, False)

root
 |-- AdmissionID: integer (nullable = true)
 |-- PatientID: integer (nullable = true)
 |-- AdmissionDate: timestamp (nullable = true)
 |-- DischargeDate: timestamp (nullable = true)
 |-- AdmissionReason: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- DoctorName: string (nullable = true)
 |-- AdmissionStatus: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

columns: 9
Rows: 974


In [5]:
## Null Profiling

from pyspark.sql.functions import col , desc , trim
null_profile_admission=[]

for c in df_admissions.columns:
    null_count=df_admissions.filter(col(c).isNull()).count()
    null_profile_admission.append((c,null_count))


null_count_admission_df= spark.createDataFrame(null_profile_admission,["columns","null_count"])
display(null_count_admission_df)

## Blank Profiling

from pyspark.sql.functions import col

blank_count_admission=[]

for c in df_admissions.columns:
    blank_count = df_admissions.filter(trim(col(c))=="").count()
    blank_count_admission.append((c,blank_count))

blank_count_admission_df=  spark.createDataFrame(blank_count_admission,["columns","blank_count"])
display(blank_count_admission_df)

## categorical columns

cat_col = ["AdmissionReason","Department","DoctorName","AdmissionStatus"]

for c in cat_col:
    print(f"\n====={c}======")

    df_admissions.groupBy(c).count().orderBy(desc("count")).show()




StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc205faa-d2bd-4961-94fb-fc1989e20e28)

SynapseWidget(Synapse.DataFrame, e7cfeddf-61fd-410c-baf2-c255ff6b2865)


=====AdmissionReason======
+---------------+-----+
|AdmissionReason|count|
+---------------+-----+
|               |  115|
|     Chest Pain|  111|
|       Accident|  102|
|   Hypertension|  101|
|       Headache|   96|
|          Fever|   92|
|       Fracture|   91|
|      Follow-up|   90|
|       Diabetes|   81|
|      Infection|   79|
|           NULL|   16|
+---------------+-----+


=====Department======
+----------------+-----+
|      Department|count|
+----------------+-----+
|      Cardiology|  118|
|       Emergency|  117|
|       neurology|  113|
|       Neurology|  109|
|     Cardiology |  104|
|     Orthopedics|  104|
|General Medicine|  100|
|      Pediatrics|   98|
|        Oncology|   98|
|            NULL|   13|
+----------------+-----+


=====DoctorName======
+-----------------+-----+
|       DoctorName|count|
+-----------------+-----+
|  Dr. Kabir Gupta|   12|
|Dr. Aditya Sharma|   12|
|  Dr. Aditi Reddy|   11|
| Dr. Aditya Patel|   10|
| Dr. Arjun Sharma|   10|
|  Dr.

In [6]:
##IDENTIFIER columns
Identifier = df_admissions.select("AdmissionID","PatientID").limit(10)

display(Identifier)


StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 66e8d7f2-fc58-4ac9-a0f8-9e39ef889f44)

In [7]:
##Check duplicates

df_admissions.groupBy("AdmissionID").count().filter(
    col("count")>1
).show()

df_admissions_clean = df_admissions.dropDuplicates()

print("Before:",df_admissions.count())
print("after:",df_admissions_clean.count())

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 9, Finished, Available, Finished, False)

+-----------+-----+
|AdmissionID|count|
+-----------+-----+
+-----------+-----+

Before: 974
after: 974


In [8]:
##date profiling and cross-column validation.

df_admissions.filter(
    col("DischargeDate").isNotNull() &
    (col("DischargeDate")<col("AdmissionDate"))
).show()


df_admissions.filter(
    col("AdmissionDate").isNull()
).show()

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 10, Finished, Available, Finished, False)

+-----------+---------+-------------------+-------------------+---------------+----------------+-----------------+---------------+-------------------+
|AdmissionID|PatientID|      AdmissionDate|      DischargeDate|AdmissionReason|      Department|       DoctorName|AdmissionStatus|   LastModifiedDate|
+-----------+---------+-------------------+-------------------+---------------+----------------+-----------------+---------------+-------------------+
|         47|      844|2025-06-01 18:56:00|2025-05-27 18:56:00|       Headache|     Orthopedics|  Dr. Rohan Patel|       Admitted|2025-06-04 15:56:00|
|         94|      103|2026-02-18 01:23:00|2026-02-17 01:23:00|          Fever|        Oncology| Dr. Kabir Kapoor|       Admitted|2026-02-18 18:23:00|
|        141|      737|2025-06-03 12:03:00|2025-06-01 12:03:00|          Fever|       neurology|  Dr. Sneha Singh|    Transferred|2025-06-04 21:03:00|
|        188|      912|2025-07-14 03:41:00|2025-07-09 03:41:00|       Headache|General Medicin

In [9]:

from pyspark.sql.functions import col , desc , count , when
df_admissions.groupBy(
    "AdmissionStatus"
).agg(
    count("*").alias("Total"),
    count(when(col("DischargeDate").isNull(), True)).alias("Null_DischargeDate")
).show()

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 11, Finished, Available, Finished, False)

+---------------+-----+------------------+
|AdmissionStatus|Total|Null_DischargeDate|
+---------------+-----+------------------+
|       admitted|  164|               160|
|           NULL|    9|                 2|
|       Admitted|  145|               140|
|     DISCHARGED|  192|                 0|
|    Transferred|  155|                 0|
|     Discharged|  161|                 0|
|               |  148|                 0|
+---------------+-----+------------------+



In [10]:
silver_admissions = df_admissions_clean

silver_admissions.printSchema()

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 12, Finished, Available, Finished, False)

root
 |-- AdmissionID: integer (nullable = true)
 |-- PatientID: integer (nullable = true)
 |-- AdmissionDate: timestamp (nullable = true)
 |-- DischargeDate: timestamp (nullable = true)
 |-- AdmissionReason: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- DoctorName: string (nullable = true)
 |-- AdmissionStatus: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [11]:
##String Operations

from pyspark.sql.functions import trim , initcap , lower 

String_col = ["AdmissionReason","Department","DoctorName","AdmissionStatus"]

for c in String_col:
    silver_admissions=silver_admissions.withColumn(c,trim(lower(col(c))))


StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 13, Finished, Available, Finished, False)

In [12]:
## Admisssion Sttsus

silver_admissions=silver_admissions.withColumn("AdmissionStatus",
initcap(trim(col("AdmissionStatus")))
)

silver_admissions.groupBy("AdmissionStatus").count().show()



StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 14, Finished, Available, Finished, False)

+---------------+-----+
|AdmissionStatus|count|
+---------------+-----+
|           NULL|    9|
|       Admitted|  309|
|    Transferred|  155|
|     Discharged|  353|
|               |  148|
+---------------+-----+



In [13]:
## Department

silver_admissions=silver_admissions.withColumn("Department", initcap(trim(col("Department")))

)
silver_admissions.groupBy("Department").count().show()

silver_admissions.filter(col("Department").isNull()).show()

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 15, Finished, Available, Finished, False)

+----------------+-----+
|      Department|count|
+----------------+-----+
|            NULL|   13|
|       Neurology|  222|
|      Cardiology|  222|
|      Pediatrics|   98|
|       Emergency|  117|
|     Orthopedics|  104|
|General Medicine|  100|
|        Oncology|   98|
+----------------+-----+

+-----------+---------+-------------------+-------------------+---------------+----------+----------------+---------------+-------------------+
|AdmissionID|PatientID|      AdmissionDate|      DischargeDate|AdmissionReason|Department|      DoctorName|AdmissionStatus|   LastModifiedDate|
+-----------+---------+-------------------+-------------------+---------------+----------+----------------+---------------+-------------------+
|         73|      407|2025-01-03 14:40:00|               NULL|       fracture|      NULL| dr. aditi reddy|       Admitted|2025-01-03 16:40:00|
|        146|      543|2025-06-03 13:49:00|2025-06-15 22:49:00|          fever|      NULL| dr. rahul patel|    Transferred|

In [14]:

silver_admissions=silver_admissions.withColumn("DoctorName", initcap(trim(col("DoctorName"))))

silver_admissions=silver_admissions.withColumn("AdmissionReason",initcap(trim(col("AdmissionReason"))))
silver_admissions.filter(
    col("AdmissionReason").isNull() |
    (trim(col("AdmissionReason")) == "")
).show()

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 16, Finished, Available, Finished, False)

+-----------+---------+-------------------+-------------------+---------------+----------------+----------------+---------------+-------------------+
|AdmissionID|PatientID|      AdmissionDate|      DischargeDate|AdmissionReason|      Department|      DoctorName|AdmissionStatus|   LastModifiedDate|
+-----------+---------+-------------------+-------------------+---------------+----------------+----------------+---------------+-------------------+
|         26|      728|2025-04-23 08:28:00|2025-04-28 18:28:00|               |      Cardiology|Dr. Vivaan Mehta|     Discharged|2025-04-23 08:28:00|
|         34|      477|2026-01-31 22:18:00|2026-02-13 23:18:00|               |General Medicine|Dr. Ishaan Reddy|     Discharged|2026-02-03 11:18:00|
|         37|       25|2026-04-05 13:46:00|2026-04-16 01:46:00|               |      Cardiology| Dr. Ishaan Nair|    Transferred|2026-04-07 01:46:00|
|         51|      416|2026-03-17 08:01:00|2026-03-26 11:01:00|               |       Emergency|Dr. 

In [15]:
### Validations 


silver_admissions.filter(
    col("PatientID").isNull()
).count()

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 17, Finished, Available, Finished, False)

0

In [16]:
silver_admissions.groupBy(
    "AdmissionStatus"
).agg(
    count("*").alias("Total"),
    count(
        when(col("DischargeDate").isNull(), 1)
    ).alias("NullDischargeDate")
).show()

silver_admissions.groupBy("AdmissionReason") \
    .count() \
    .orderBy(desc("count")) \
    .show(truncate=False)


silver_admissions.filter(
    (col("AdmissionStatus") == "Admitted") &
    col("DischargeDate").isNotNull()
).show(truncate=False)

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 18, Finished, Available, Finished, False)

+---------------+-----+-----------------+
|AdmissionStatus|Total|NullDischargeDate|
+---------------+-----+-----------------+
|           NULL|    9|                2|
|       Admitted|  309|              300|
|    Transferred|  155|                0|
|     Discharged|  353|                0|
|               |  148|                0|
+---------------+-----+-----------------+

+---------------+-----+
|AdmissionReason|count|
+---------------+-----+
|               |115  |
|Chest Pain     |111  |
|Accident       |102  |
|Hypertension   |101  |
|Headache       |96   |
|Fever          |92   |
|Fracture       |91   |
|Follow-up      |90   |
|Diabetes       |81   |
|Infection      |79   |
|NULL           |16   |
+---------------+-----+

+-----------+---------+-------------------+-------------------+---------------+----------------+----------------+---------------+-------------------+
|AdmissionID|PatientID|AdmissionDate      |DischargeDate      |AdmissionReason|Department      |DoctorName    

In [17]:
from pyspark.sql.functions import col, when, concat_ws, lit, trim

silver_admissions = silver_admissions.withColumn(
    "DataQualityReason",
    concat_ws(
        "; ",

        when(
            col("AdmissionDate").isNull(),
            lit("Missing AdmissionDate")
        ),

        when(
            col("DischargeDate").isNotNull() &
            (col("DischargeDate") < col("AdmissionDate")),
            lit("DischargeDate before AdmissionDate")
        ),

        when(
            col("Department").isNull() |
            (trim(col("Department")) == ""),
            lit("Missing Department")
        ),

        when(
            col("AdmissionReason").isNull() |
            (trim(col("AdmissionReason")) == ""),
            lit("Missing AdmissionReason")
        ),

        when(
            col("AdmissionStatus").isNull() |
            (trim(col("AdmissionStatus")) == ""),
            lit("Missing AdmissionStatus")
        ),

        when(
            (col("AdmissionStatus") == "Discharged") &
            col("DischargeDate").isNull(),
            lit("Discharged patient missing DischargeDate")
        ),

        when(
            (col("AdmissionStatus") == "Transferred") &
            col("DischargeDate").isNull(),
            lit("Transferred patient missing DischargeDate")
        ),

        when(
            (col("AdmissionStatus") == "Admitted") &
            col("DischargeDate").isNotNull(),
            lit("Admitted patient has DischargeDate")
        )
    )
)
silver_admissions.groupBy("DataQualityReason") \
    .count() \
    .show(truncate=False)

display(silver_admissions)


StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 19, Finished, Available, Finished, False)

+-----------------------------------------------------------------------------------------------+-----+
|DataQualityReason                                                                              |count|
+-----------------------------------------------------------------------------------------------+-----+
|DischargeDate before AdmissionDate                                                             |8    |
|Missing AdmissionReason; Missing AdmissionStatus                                               |14   |
|Missing Department; Missing AdmissionStatus                                                    |2    |
|Missing AdmissionDate; Missing AdmissionReason                                                 |1    |
|Missing AdmissionDate; Missing AdmissionStatus                                                 |1    |
|DischargeDate before AdmissionDate; Admitted patient has DischargeDate                         |8    |
|Missing Department                                             

SynapseWidget(Synapse.DataFrame, 3bbc1192-570d-4d04-9059-f178d677018c)

In [18]:
from pyspark.sql.functions import col

silver_admissions_valid = silver_admissions.filter(
    col("DataQualityReason") == ""
)

silver_admissions_review = silver_admissions.filter(
    col("DataQualityReason") != ""
)

display(silver_admissions_valid)

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c285bac-f09f-40b1-a895-985dd4748c21)

In [19]:
print("Total records:", silver_admissions.count())
print("Valid records:", silver_admissions_valid.count())
print("Review records:", silver_admissions_review.count())

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 21, Finished, Available, Finished, False)

Total records: 974
Valid records: 668
Review records: 306


In [20]:

## OLD INITIAL LOAD LOGIC

#silver_admissions_valid.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("silver_admissions_valid")

#silver_admissions_review.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("silver_admissions_review")

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 22, Finished, Available, Finished, False)

### Cross Table validation (patients id which are present in admission but not in patients)

In [21]:
patient_valid = spark.table("silver_patients")
patient_review = spark.table("silver_patients_review")

admission_valid = silver_admissions_valid
admission_review = silver_admissions_review


all_patient_id = (
    patient_valid.select("PatientID")
    .union(
        patient_review.select("PatientID")
    ).distinct()
)

print(f"Total Patient Id:",all_patient_id.count())

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 23, Finished, Available, Finished, False)

Total Patient Id: 992


In [22]:
from pyspark.sql.functions import col, lit

orphan_admissions = admission_valid.join(
    all_patient_id,on="PatientID",how="left_anti"
)
print("Orphan Admission:",orphan_admissions.count())

orphan_admissions=orphan_admissions.withColumn(
    "DataQualityReason",
    lit("Invalid PatientID - Patient not found")
)

orphan_admissions.select(
    "AdmissionID",
    "PatientID",
    "DataQualityReason"
).show(truncate=False)



StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 24, Finished, Available, Finished, False)

Orphan Admission: 4
+-----------+---------+-------------------------------------+
|AdmissionID|PatientID|DataQualityReason                    |
+-----------+---------+-------------------------------------+
|111        |113      |Invalid PatientID - Patient not found|
|187        |339      |Invalid PatientID - Patient not found|
|639        |565      |Invalid PatientID - Patient not found|
|659        |791      |Invalid PatientID - Patient not found|
+-----------+---------+-------------------------------------+



In [23]:
admission_valid_updated = admission_valid.join(
    orphan_admissions.select("AdmissionID"),
    on="AdmissionID",
    how="left_anti"
)


print("Before:", admission_valid.count())
print("Orphans:", orphan_admissions.count())
print("After:", admission_valid_updated.count())

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 25, Finished, Available, Finished, False)

Before: 668
Orphans: 4
After: 664


In [24]:
admission_review_updated = admission_review.unionByName(
    orphan_admissions,
    allowMissingColumns=True
)

print("Valid Admissions:", admission_valid_updated.count())
print("Review Admissions:", admission_review_updated.count())




StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 26, Finished, Available, Finished, False)

Valid Admissions: 664
Review Admissions: 310


In [25]:
admission_valid_updated.createOrReplaceTempView(
    "new_valid_admissions"
)

spark.sql("""

MERGE INTO silver_admissions_valid as TARGET
USING new_valid_admissions as SOURCE
ON target.AdmissionID = source.AdmissionID

WHEN MATCHED THEN UPDATE SET *

WHEN  NOT MATCHED THEN INSERT *

""")

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 27, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [26]:
admission_review_updated.createOrReplaceTempView(

    "new_review_admissions"
)

spark.sql("""

MERGE INTO silver_admissions_review as TARGET
USING new_review_admissions AS SOURCE
ON TARGET.AdmissionID = SOURCE.AdmissionID

when MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *

""")

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 28, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [27]:
# old load logic

#admission_valid_updated.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("silver_admissions_valid")

#admission_review_updated.write \
 #   .format("delta") \
  #  .mode("overwrite") \
   # .saveAsTable("silver_admissions_review")

StatementMeta(, 5a084284-ea48-46e5-a5ac-68d1a7d030ee, 29, Finished, Available, Finished, False)